# MMMU-Pro (Vision) headroom analysis with Qwen2.5-VL-7B-Instruct

**Goal:** Estimate the headroom remaining on the MMMU-Pro *vision* benchmark for a strong open multimodal-reasoning model, and construct a curated set of **wrong predictions attributable to multimodal issues** (perception / OCR / spatial / chart-reading failures) rather than pure knowledge or reasoning gaps.

**Setup**
- Model: `Qwen/Qwen2.5-VL-7B-Instruct` (best open VLM that fits a 24 GB RTX 4090 in bf16).
- Benchmark: `MMMU/MMMU_Pro`, config `vision` — the question **and** answer options are rendered *into a screenshot*; the model must read everything visually. This maximally stresses true multimodal reading, so failures concentrate the multimodal error modes we want to study.
- Scope: ~200 samples (configurable via `N_SAMPLES`).

**Pipeline:** load data → CoT inference → parse `Answer: X` → score / headroom → categorize errors → dump multimodal-failure set.

In [1]:
import os, re, json, time, random
os.environ.setdefault("HF_HOME", "/workspace/.hf_home")
import torch
from datasets import load_dataset
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# ---- config ----
MODEL_ID    = "Qwen/Qwen2.5-VL-7B-Instruct"
N_SAMPLES   = 200          # vision-set samples to evaluate
MAX_NEW_TOK = 1024         # room for chain-of-thought + final "Answer: X"
SEED        = 0
OUT_DIR     = "/workspace/mmmu_pro_out"
os.makedirs(OUT_DIR, exist_ok=True)
random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda"
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0))

[ERROR] `loss` is part of Qwen2_5_VLCausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /venv/main/lib/python3.14/site-packages/transformers/models/qwen2_5_vl/modeling_qwen2_5_vl.py.
[ERROR] `logits` is part of Qwen2_5_VLCausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /venv/main/lib/python3.14/site-packages/transformers/models/qwen2_5_vl/modeling_qwen2_5_vl.py.
torch 2.11.0+cu130 | cuda True | NVIDIA GeForce RTX 4090


In [2]:
# ---- load MMMU-Pro vision split ----
ds = load_dataset("MMMU/MMMU_Pro", "vision", split="test")
print("full vision split size:", len(ds))
print("features:", {k: str(v) for k, v in ds.features.items()})

ex = ds[0]
for k, v in ex.items():
    print(f"  {k:12s}:", (f"PIL {v.size} {v.mode}" if hasattr(v, "size") else repr(v)[:160]))

# Deterministic subsample of N_SAMPLES
idx = list(range(len(ds)))
random.Random(SEED).shuffle(idx)
sel = sorted(idx[:N_SAMPLES])
subset = ds.select(sel)
print("\nEvaluating", len(subset), "samples")

full vision split size: 1730
features: {'id': "Value('string')", 'image': 'Image(mode=None, decode=True)', 'options': "Value('string')", 'answer': "Value('string')", 'subject': "Value('string')"}
  id          : 'test_History_1'
  image       : PIL (1014, 1182) RGBA
  options     : "['Political instability leading to population decline', 'The spread of pathogens across the Silk Road', 'Development of new trade routes', 'Climate change affe
  answer      : 'B'
  subject     : 'History'

Evaluating 200 samples


In [3]:
# ---- load model + processor ----
t0 = time.time()
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map=DEVICE,
).eval()
# Cap pixel budget so big screenshots don't blow up the visual token count / VRAM.
processor = AutoProcessor.from_pretrained(
    MODEL_ID, min_pixels=256*28*28, max_pixels=1280*28*28,
)
print(f"loaded in {time.time()-t0:.1f}s | VRAM {torch.cuda.memory_allocated()/1e9:.1f} GB")

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

loaded in 4.1s | VRAM 16.6 GB


In [4]:
# ---- prompt + inference ----
# In the vision setting the question and ALL options are rendered into the screenshot,
# so the text prompt only carries the answer-format instruction (official MMMU-Pro style).
VISION_PROMPT = (
    "The image contains a multiple-choice question together with its answer options. "
    "Read the question and every option carefully, then reason step by step to solve it. "
    "The last line of your response must be exactly of the form 'Answer: $LETTER' "
    "where $LETTER is the single letter (A, B, C, ...) of the correct option."
)

def to_rgb(image):
    # MMMU-Pro vision screenshots are RGBA; flatten onto white so the VLM sees clean text.
    if image.mode == "RGB":
        return image
    from PIL import Image as _Img
    if image.mode in ("RGBA", "LA", "P"):
        im = image.convert("RGBA")
        bg = _Img.new("RGB", im.size, (255, 255, 255))
        bg.paste(im, mask=im.split()[-1])
        return bg
    return image.convert("RGB")

ANS_RE = re.compile(r"Answer:\s*\(?([A-J])\)?", re.IGNORECASE)

def parse_answer(text: str):
    """Extract the final answer letter; prefer the last 'Answer: X', else last bare letter."""
    m = list(ANS_RE.finditer(text))
    if m:
        return m[-1].group(1).upper()
    # fallback: last standalone capital letter A-J in the text
    m2 = list(re.finditer(r"\b([A-J])\b", text))
    return m2[-1].group(1).upper() if m2 else None

@torch.inference_mode()
def generate(image, prompt, max_new_tokens):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": to_rgb(image)},
        {"type": "text",  "text": prompt},
    ]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors="pt").to(DEVICE)
    gen = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    trimmed = gen[:, inputs.input_ids.shape[1]:]
    return processor.batch_decode(trimmed, skip_special_tokens=True,
                                  clean_up_tokenization_spaces=False)[0]

def run_one(image):
    return generate(image, VISION_PROMPT, MAX_NEW_TOK)

# smoke test on one sample after the model loads
_t = run_one(subset[0]["image"])
print(_t[-400:])
print("--> parsed:", parse_answer(_t), "| gold:", subset[0]["answer"])

y often causes yellowing of the leaves, especially at the margins.
- The stunted growth could also be indicative of nitrogen deficiency.
- No other specific nutrient deficiencies (like zinc, potassium, magnesium, molybdenum, calcium, sulfur, iron, or phosphate) typically cause such a uniform yellowing pattern without stunting.

Therefore, the most likely nutrient deficiency is nitrogen.

Answer: I
--> parsed: I | gold: J


In [5]:
# ---- run inference over the subset ----
results = []
t0 = time.time()
for i, ex in enumerate(subset):
    out = run_one(ex["image"])
    pred = parse_answer(out)
    gold = str(ex["answer"]).strip().upper()
    results.append({
        "i": i,
        "id": ex.get("id", i),
        "subject": ex.get("subject"),
        "gold": gold,
        "pred": pred,
        "correct": (pred == gold),
        "options": ex.get("options", None),
        "raw": out,
    })
    if (i + 1) % 10 == 0 or i == 0:
        acc = sum(r["correct"] for r in results) / len(results)
        el = time.time() - t0
        msg = (f"[{i+1:3d}/{len(subset)}] running acc={acc:.3f} | "
               f"{el:.0f}s ({el/(i+1):.1f}s/ex)")
        print(msg, flush=True)
        # checkpoint (crash-safety + live progress)
        with open(f"{OUT_DIR}/raw_results.json", "w") as f:
            json.dump(results, f, indent=2)
        with open(f"{OUT_DIR}/progress.log", "a") as f:
            f.write(msg + "\n")

with open(f"{OUT_DIR}/raw_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("saved", f"{OUT_DIR}/raw_results.json")

[  1/200] running acc=0.000 | 4s (3.7s/ex)


[ 10/200] running acc=0.300 | 57s (5.7s/ex)


[ 20/200] running acc=0.400 | 139s (6.9s/ex)


[ 30/200] running acc=0.367 | 222s (7.4s/ex)


[ 40/200] running acc=0.375 | 316s (7.9s/ex)


...

[170/200] running acc=0.335 | 1432s (8.4s/ex)


[180/200] running acc=0.339 | 1502s (8.3s/ex)


[190/200] running acc=0.332 | 1568s (8.3s/ex)


[200/200] running acc=0.320 | 1636s (8.2s/ex)


saved /workspace/mmmu_pro_out/raw_results.json


In [6]:
# ---- accuracy + headroom ----
n = len(results)
n_correct = sum(r["correct"] for r in results)
n_unparsed = sum(r["pred"] is None for r in results)
acc = n_correct / n

# Published reference points for MMMU-Pro (vision) to frame headroom:
REFS = {
    "Qwen2.5-VL-7B (reported, vision)": 0.49,   # approx, Qwen2.5-VL tech report
    "Qwen2.5-VL-72B (reported, vision)": 0.59,
    "GPT-4o (reported, vision)":         0.49,
    "Frontier reasoning (o1-class)":     0.65,
    "Human expert (MMMU-Pro est.)":      0.90,
}

print(f"N evaluated         : {n}")
print(f"Correct             : {n_correct}")
print(f"Unparsed predictions: {n_unparsed}")
print(f"Accuracy            : {acc:.3f}")
import math
se = math.sqrt(acc*(1-acc)/n)
print(f"95% CI              : [{acc-1.96*se:.3f}, {acc+1.96*se:.3f}]")
print("\nHeadroom vs reference points:")
for k, v in REFS.items():
    print(f"  {k:34s} {v:.2f}   gap = {v-acc:+.3f}")
print(f"\nWrong predictions to analyze: {n - n_correct}")

N evaluated         : 200
Correct             : 64
Unparsed predictions: 2
Accuracy            : 0.320
95% CI              : [0.255, 0.385]

Headroom vs reference points:
  Qwen2.5-VL-7B (reported, vision)   0.49   gap = +0.170
  Qwen2.5-VL-72B (reported, vision)  0.59   gap = +0.270
  GPT-4o (reported, vision)          0.49   gap = +0.170
  Frontier reasoning (o1-class)      0.65   gap = +0.330
  Human expert (MMMU-Pro est.)       0.90   gap = +0.580

Wrong predictions to analyze: 136


In [7]:
# ---- diagnose WHY each wrong prediction failed ----
# Two complementary signals per wrong item:
#  (1) OCR-faithfulness probe (ground-truth-anchored): ask the model to transcribe the
#      options straight from the screenshot, fuzzy-match vs the gold `options` text.
#      Low match => the model literally misread the choices => a perception/OCR failure.
#  (2) Image-grounded judge: show image + gold + the model's own reasoning, classify the
#      failure into a taxonomy and flag whether it is multimodal-attributable.
from difflib import SequenceMatcher
import ast

TAXONOMY = ["OCR_TEXT", "CHART_PLOT", "TABLE", "DIAGRAM_SPATIAL",
            "FINE_DETAIL", "COUNTING", "REASONING", "KNOWLEDGE", "FORMAT_PARSING"]
MULTIMODAL_CATS = {"OCR_TEXT", "CHART_PLOT", "TABLE", "DIAGRAM_SPATIAL",
                   "FINE_DETAIL", "COUNTING"}

def _norm(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())

def gold_options(ex):
    o = ex.get("options")
    if isinstance(o, str):
        try: o = ast.literal_eval(o)
        except Exception: o = [o]
    return list(o) if o else []

OCR_PROMPT = (
    "Transcribe ONLY the multiple-choice answer options exactly as they appear in the "
    "image, one per line, each prefixed by its letter like 'A: ...'. Do not solve the question."
)

def ocr_probe(image):
    return generate(image, OCR_PROMPT, 400)

def ocr_faithfulness(transcript, gold_opts):
    """Best fuzzy match of each gold option against any transcribed line; return mean ratio."""
    if not gold_opts:
        return None
    lines = [l for l in transcript.splitlines() if l.strip()]
    tnorm = [_norm(l) for l in lines]
    scores = []
    for go in gold_opts:
        gn = _norm(go)
        if not gn:
            continue
        best = max((SequenceMatcher(None, gn, t).ratio() for t in tnorm), default=0.0)
        scores.append(best)
    return sum(scores) / len(scores) if scores else None

JUDGE_TMPL = (
    "You are auditing why a vision-language model got a multiple-choice question WRONG. "
    "The question and its options are rendered inside the image shown.\n"
    "GOLD answer: {gold}\nMODEL answer: {pred}\n"
    "MODEL's reasoning was:\n\"\"\"\n{trace}\n\"\"\"\n\n"
    "First, look at the image yourself and verify what it actually shows. Then decide the "
    "PRIMARY cause of the model's error, choosing exactly one category:\n"
    "- OCR_TEXT: misread/garbled the question or option TEXT\n"
    "- CHART_PLOT: misread values/axes/trends in a chart, graph or plot\n"
    "- TABLE: misread tabular data\n"
    "- DIAGRAM_SPATIAL: misread spatial/geometric/structural relations or a diagram\n"
    "- FINE_DETAIL: missed small / low-contrast / partially-occluded visual detail\n"
    "- COUNTING: miscounted objects/items\n"
    "- REASONING: read the image correctly but reasoned/calculated incorrectly\n"
    "- KNOWLEDGE: lacked the domain knowledge regardless of perception\n"
    "- FORMAT_PARSING: answer was correct in spirit but mis-formatted / unparseable\n\n"
    "Respond with ONLY a JSON object: "
    '{{"category": "<one>", "multimodal": <true|false>, '
    '"confidence": <0-1>, "evidence": "<=25 words"}}. '
    "Set multimodal=true iff the primary cause is a failure to correctly PERCEIVE the image."
)

JSON_RE = re.compile(r"\{.*\}", re.DOTALL)

def judge(image, gold, pred, trace):
    out = generate(image, JUDGE_TMPL.format(gold=gold, pred=pred, trace=trace[:2000]), 200)
    m = JSON_RE.search(out)
    if not m:
        return {"category": "UNKNOWN", "multimodal": None, "confidence": 0, "evidence": out[:120]}
    try:
        j = json.loads(m.group(0))
        if j.get("category") not in TAXONOMY:
            j["category"] = "UNKNOWN"
        return j
    except Exception:
        return {"category": "UNKNOWN", "multimodal": None, "confidence": 0, "evidence": out[:120]}

print("diagnostic functions ready")

diagnostic functions ready


In [8]:
# ---- run diagnosis over wrong predictions ----
OCR_FAIL_THR = 0.55   # mean option-transcription fuzzy ratio below this => OCR/perception failure

wrong = [r for r in results if not r["correct"]]
print(f"diagnosing {len(wrong)} wrong predictions...", flush=True)

diagnosed = []
for k, r in enumerate(wrong):
    ex = subset[r["i"]]
    img = ex["image"]
    g_opts = gold_options(ex)
    transcript = ocr_probe(img)
    ocr_score = ocr_faithfulness(transcript, g_opts)
    jr = judge(img, r["gold"], r["pred"], r["raw"])

    cat = jr.get("category", "UNKNOWN")
    judge_mm = bool(jr.get("multimodal"))
    ocr_mm = (ocr_score is not None and ocr_score < OCR_FAIL_THR)
    # multimodal-attributable if EITHER the ground-truthed OCR probe shows misreading,
    # OR the judge attributes it to perception (category in the multimodal set).
    is_multimodal = bool(ocr_mm or judge_mm or (cat in MULTIMODAL_CATS))

    diagnosed.append({**r,
        "category": cat,
        "judge_multimodal": judge_mm,
        "ocr_faithfulness": ocr_score,
        "ocr_misread": ocr_mm,
        "is_multimodal": is_multimodal,
        "confidence": jr.get("confidence"),
        "evidence": jr.get("evidence"),
        "ocr_transcript": transcript,
    })
    if (k + 1) % 5 == 0 or k == 0:
        msg = f"  [{k+1}/{len(wrong)}] last: cat={cat} mm={is_multimodal} ocr={ocr_score}"
        print(msg, flush=True)
        with open(f"{OUT_DIR}/diagnosed_wrong.json", "w") as f:
            json.dump(diagnosed, f, indent=2)
        with open(f"{OUT_DIR}/progress.log", "a") as f:
            f.write(msg + "\n")

with open(f"{OUT_DIR}/diagnosed_wrong.json", "w") as f:
    json.dump(diagnosed, f, indent=2)
print("saved", f"{OUT_DIR}/diagnosed_wrong.json")

diagnosing 136 wrong predictions...


  [1/136] last: cat=REASONING mm=False ocr=0.9350982795786814


  [5/136] last: cat=REASONING mm=False ocr=0.888888888888889


  [10/136] last: cat=REASONING mm=False ocr=0.9685664663601502


  [15/136] last: cat=REASONING mm=False ocr=0.960409763886131


...

  [120/136] last: cat=REASONING mm=False ocr=0.9325514644577007


  [125/136] last: cat=REASONING mm=False ocr=0.9733572037341084


  [130/136] last: cat=REASONING mm=False ocr=0.9686442402189911


  [135/136] last: cat=REASONING mm=False ocr=0.9842913378099658


saved /workspace/mmmu_pro_out/diagnosed_wrong.json


In [9]:
# ---- summary + curated multimodal-failure set ----
from collections import Counter

cats = Counter(d["category"] for d in diagnosed)
mm_set = [d for d in diagnosed if d["is_multimodal"]]

print("="*64)
print("MMMU-Pro (vision) — Qwen2.5-VL-7B-Instruct")
print("="*64)
print(f"Accuracy                : {acc:.3f}  ({n_correct}/{n})")
print(f"Total wrong             : {len(wrong)}")
print(f"Multimodal-attributable : {len(mm_set)}  "
      f"({len(mm_set)/max(len(wrong),1):.0%} of errors, "
      f"{len(mm_set)/n:.0%} of all items)")
print("\nError category breakdown (of wrong preds):")
for c, v in cats.most_common():
    tag = "  <-- multimodal" if c in MULTIMODAL_CATS else ""
    print(f"  {c:16s} {v:3d}{tag}")

# Headroom interpretation
recoverable = len(mm_set) / n
print(f"\nHeadroom attributable to multimodal/perception issues: ~{recoverable:.0%} "
      f"of items (i.e. fixing perception alone could lift accuracy from "
      f"{acc:.2f} toward {acc+recoverable:.2f}).")

# Save the curated set (drop bulky raw fields for the headline export)
curated = [{
    "id": d["id"], "gold": d["gold"], "pred": d["pred"],
    "category": d["category"], "is_multimodal": d["is_multimodal"],
    "judge_multimodal": d["judge_multimodal"], "ocr_misread": d["ocr_misread"],
    "ocr_faithfulness": d["ocr_faithfulness"], "confidence": d["confidence"],
    "evidence": d["evidence"],
} for d in mm_set]
with open(f"{OUT_DIR}/multimodal_failures.json", "w") as f:
    json.dump({"summary": {"accuracy": acc, "n": n, "n_wrong": len(wrong),
                           "n_multimodal": len(mm_set),
                           "categories": dict(cats)},
               "cases": curated}, f, indent=2)

# Export up to 12 example screenshots for visual inspection
ex_dir = f"{OUT_DIR}/multimodal_examples"; os.makedirs(ex_dir, exist_ok=True)
for d in mm_set[:12]:
    subset[d["i"]]["image"].save(f"{ex_dir}/{d['id']}_{d['category']}_gold{d['gold']}_pred{d['pred']}.png")

print(f"\nSaved: {OUT_DIR}/multimodal_failures.json  (+ {min(12,len(mm_set))} example PNGs)")
print("\n--- sample multimodal failures ---")
for d in mm_set[:6]:
    print(f"[{d['id']}] {d['category']} gold={d['gold']} pred={d['pred']} "
          f"ocr={d['ocr_faithfulness']} :: {d['evidence']}")

MMMU-Pro (vision) — Qwen2.5-VL-7B-Instruct
Accuracy                : 0.320  (64/200)
Total wrong             : 136
Multimodal-attributable : 24  (18% of errors, 12% of all items)

Error category breakdown (of wrong preds):
  REASONING        114
  DIAGRAM_SPATIAL   13  <-- multimodal
  COUNTING           3  <-- multimodal
  TABLE              2  <-- multimodal
  UNKNOWN            2
  CHART_PLOT         1  <-- multimodal
  FORMAT_PARSING     1

Headroom attributable to multimodal/perception issues: ~12% of items (i.e. fixing perception alone could lift accuracy from 0.32 toward 0.44).



Saved: /workspace/mmmu_pro_out/multimodal_failures.json  (+ 12 example PNGs)

--- sample multimodal failures ---
[test_Economics_182] TABLE gold=C pred=A ocr=0.9913681646647942 :: The model incorrectly calculated the marginal product of the third worker as 5 pots instead of 3 pots.
[validation_Pharmacy_11] REASONING gold=A pred=C ocr=0.5 :: The model correctly identified that the amplitude is the height of the peaks and chose Wave C as having the highest amplitude.
[test_Biology_157] DIAGRAM_SPATIAL gold=I pred=J ocr=0.9659814263910805 :: The model misidentified the structure labeled 210 as the Jugular foramen instead of Foramen lacerum due to incorrect spatial interpretation.
[validation_Biology_1] DIAGRAM_SPATIAL gold=D pred=B ocr=0.9618496857049392 :: The model misidentified the location of the Soleus muscle due to incorrect spatial understanding of the posterior compartment.
[validation_Biology_22] DIAGRAM_SPATIAL gold=J pred=H ocr=0.7866666666666667 :: The model misinterpreted th